# Gradient descent sandbox: vanilla GD vs momentum

A small playground that goes with the 15-minute "Introduction to gradient descent" lecture.

You have two optimizers operating on a 1D loss surface $L(\theta)$:

**Vanilla gradient descent**

$$\theta_{t+1} = \theta_t - \eta \, \nabla L(\theta_t)$$

**Gradient descent with (classical) momentum**

$$v_{t+1} = \beta \, v_t + \nabla L(\theta_t)$$
$$\theta_{t+1} = \theta_t - \eta \, v_{t+1}$$

The questions to play with:
1. What value of $\eta$ makes vanilla GD diverge?
2. What value of $\beta$ lets momentum escape the local minimum?
3. What happens if $\beta$ is too high?
4. Can you find a starting point where vanilla GD doesn't get stuck?

Honest caveat: this is 1D. Real neural networks live in $\mathbb{R}^{10^9}$, where most of momentum's value is about navigating ill-conditioned landscapes, not escaping local minima. But the inertia intuition transfers cleanly, and that's what we're building here.


## 1. Setup

Just numpy and matplotlib. If you're in Colab or Jupyter, the sliders need `ipywidgets` (usually pre-installed).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed


## 2. Loss surfaces

Each surface is defined as a Python function `L(x)` plus its derivative `dL(x)`.

Add your own — just make sure the derivative is correct (or use a numerical gradient).


In [ ]:
# --- 1. Local minimum + global minimum (the lecture demo) ---
# Vanilla GD parks in the local well; momentum hops over the small bump.
def L_local_global(x):
    return (
        - 0.5 * x
        - 0.7 * np.exp(-0.5 * ((x - 2.2) / 0.45) ** 2)
        + 0.25 * np.exp(-0.5 * ((x - 3.4) / 0.30) ** 2)
        - 1.5 * np.exp(-0.5 * ((x - 5.5) / 0.70) ** 2)
        + 0.05 * (x - 4) ** 2
    )

def dL_local_global(x):
    return (
        - 0.5
        - 0.7 * np.exp(-0.5 * ((x - 2.2) / 0.45) ** 2)  * (-(x - 2.2) / 0.45 ** 2)
        + 0.25 * np.exp(-0.5 * ((x - 3.4) / 0.30) ** 2) * (-(x - 3.4) / 0.30 ** 2)
        - 1.5 * np.exp(-0.5 * ((x - 5.5) / 0.70) ** 2)  * (-(x - 5.5) / 0.70 ** 2)
        + 0.10 * (x - 4)
    )

# --- 2. Simple convex bowl ---
# Both algorithms converge; play with eta to find where vanilla GD diverges.
def L_bowl(x):
    return 0.5 * (x - 3) ** 2

def dL_bowl(x):
    return (x - 3)

# --- 3. Long shallow plateau into a well ---
# The "rolling ball accelerates on the plateau" picture.
# Vanilla GD crawls; momentum builds speed and overshoots.
def L_plateau(x):
    plateau = 1.5 - 0.10 * x
    well    = 0.20 * (x - 5.5) ** 2 - 1.5
    k = 4.0
    return -np.log(np.exp(-k * plateau) + np.exp(-k * well)) / k

def dL_plateau(x):
    plateau = 1.5 - 0.10 * x
    well    = 0.20 * (x - 5.5) ** 2 - 1.5
    dpl, dwl = -0.10, 0.40 * (x - 5.5)
    k = 4.0
    e1, e2 = np.exp(-k * plateau), np.exp(-k * well)
    return (e1 * dpl + e2 * dwl) / (e1 + e2)

# --- 4. Wiggly cosine (many local minima) ---
# Stress-test: lots of bumps. Try this with stochastic noise (later in this notebook).
def L_wiggly(x):
    return 0.1 * (x - 3) ** 2 + 0.5 * np.cos(3 * x)

def dL_wiggly(x):
    return 0.2 * (x - 3) - 1.5 * np.sin(3 * x)


SURFACES = {
    "local + global min":  (L_local_global, dL_local_global, (-1.0, 8.5), -0.5),
    "convex bowl":         (L_bowl,         dL_bowl,         (-1.0, 7.0),  6.0),
    "plateau into well":   (L_plateau,      dL_plateau,      (-1.0, 8.5), -0.5),
    "wiggly cosine":       (L_wiggly,       dL_wiggly,       (-2.0, 8.0), -1.0),
}


## 3. The two algorithms

This is the entire substance of gradient descent and momentum. Twelve lines total.

**Note on the gradient.** In a real neural network, computing $\nabla L$ requires the chain rule applied across every layer — that's *backpropagation*. In this notebook we cheat: we have the analytical derivative, so we just call `dL(x)`. The optimization logic below is identical whether the gradient comes from autograd, backprop, or pen-and-paper calculus.


In [ ]:
def vanilla_gd(dL, x0, eta, n_steps):
    """Vanilla gradient descent."""
    n_steps = int(n_steps)
    path = [x0]
    x = x0
    for _ in range(n_steps):
        x = x - eta * dL(x)
        path.append(x)
    return np.array(path)


def momentum_gd(dL, x0, eta, beta, n_steps):
    """Gradient descent with classical momentum."""
    n_steps = int(n_steps)
    path = [x0]
    x = x0
    v = 0.0
    for _ in range(n_steps):
        g = dL(x)
        v = beta * v + g
        x = x - eta * v
        path.append(x)
    return np.array(path)


## 4. Plotting

A single function that renders both trajectories on the chosen surface.


In [ ]:
def plot_run(surface_name, eta_gd, eta_mom, beta, x_start, n_steps):
    L, dL, (xmin, xmax), _default_start = SURFACES[surface_name]

    xs = np.linspace(xmin, xmax, 800)
    ys = L(xs)

    gd_path  = vanilla_gd(dL, x_start, eta_gd, n_steps)
    mom_path = momentum_gd(dL, x_start, eta_mom, beta, n_steps)

    # Clip paths that diverge so the plot doesn't blow up
    gd_path  = np.clip(gd_path,  xmin - 1, xmax + 1)
    mom_path = np.clip(mom_path, xmin - 1, xmax + 1)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(xs, ys, color="gray", lw=2, label="loss L(θ)")
    ax.plot(gd_path,  L(gd_path),  "D-", color="#D4854A",
            markersize=4, lw=1, alpha=0.7, label="vanilla GD")
    ax.plot(mom_path, L(mom_path), "o-", color="#00C9B7",
            markersize=4, lw=1, alpha=0.7, label="GD + momentum")

    ax.plot(gd_path[-1],  L(gd_path[-1]),  "D", color="#D4854A",
            markersize=12, markeredgecolor="white", mew=1.2)
    ax.plot(mom_path[-1], L(mom_path[-1]), "o", color="#00C9B7",
            markersize=14, markeredgecolor="white", mew=1.2)

    ax.set_xlabel("θ"); ax.set_ylabel("L(θ)")
    ax.set_title(
        f"{surface_name}   |   "
        f"GD final: θ={gd_path[-1]:.3f}, L={L(gd_path[-1]):.3f}   "
        f"Mom final: θ={mom_path[-1]:.3f}, L={L(mom_path[-1]):.3f}"
    )
    ax.legend(loc="upper right")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


## 5. Interactive sandbox

Move the sliders to see how vanilla GD and momentum behave on each surface.

Suggested experiments:
- **`local + global min`**: start at $\theta=-0.5$, $\eta_\text{mom}=0.10$, $\beta=0.85$. Then try lowering $\beta$ to 0.5 — momentum starts behaving like vanilla GD.
- **`convex bowl`**: try $\eta_\text{GD}=0.5$ — convergence. Then try $\eta_\text{GD}=2.5$ — divergence. The threshold is exactly $\eta=2/\text{curvature}$, here 2.
- **`plateau into well`**: notice how vanilla GD crawls forever while momentum overshoots and oscillates back.
- **`wiggly cosine`**: both algorithms get stuck. This is what stochastic noise (SGD) is for — see section 6.


In [ ]:
interact(
    plot_run,
    surface_name=Dropdown(options=list(SURFACES.keys()),
                          value="local + global min",
                          description="Surface"),
    eta_gd  = FloatSlider(value=0.20, min=0.01, max=2.0, step=0.01,
                          description="η (GD)"),
    eta_mom = FloatSlider(value=0.10, min=0.01, max=2.0, step=0.01,
                          description="η (mom)"),
    beta    = FloatSlider(value=0.85, min=0.0,  max=0.99, step=0.01,
                          description="β"),
    x_start = FloatSlider(value=-0.5, min=-2.0, max=8.0,  step=0.1,
                          description="θ_start"),
    n_steps = IntSlider(value=120, min=20, max=500, step=10,
                          description="steps"),
);


## 6. Bonus: stochastic gradient descent

Real machine learning rarely uses the *exact* gradient. With millions of training examples, computing $\nabla L$ over the whole dataset is too expensive every step. Instead we sample a *minibatch* and get a noisy estimate of the gradient.

Here we simulate that simply: add Gaussian noise to the gradient at every step.

The lesson: a bit of noise can actually *help* - it can knock the optimizer out of shallow local minima.


In [ ]:
def sgd(dL, x0, eta, n_steps, noise_std=0.0, seed=0):
    """Vanilla GD with Gaussian noise added to the gradient (poor man's SGD)."""
    n_steps = int(n_steps)
    seed = int(seed)
    rng = np.random.default_rng(seed)
    path = [x0]
    x = x0
    for _ in range(n_steps):
        g = dL(x) + rng.normal(0.0, noise_std)
        x = x - eta * g
        path.append(x)
    return np.array(path)


def plot_sgd(surface_name, eta, x_start, noise_std, n_steps, seed):
    L, dL, (xmin, xmax), _ = SURFACES[surface_name]
    xs = np.linspace(xmin, xmax, 800)
    ys = L(xs)

    gd_path  = vanilla_gd(dL, x_start, eta, n_steps)
    sgd_path = sgd(dL, x_start, eta, n_steps, noise_std=noise_std, seed=seed)
    gd_path  = np.clip(gd_path,  xmin - 1, xmax + 1)
    sgd_path = np.clip(sgd_path, xmin - 1, xmax + 1)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(xs, ys, color="gray", lw=2)
    ax.plot(gd_path,  L(gd_path),  "D-", color="#D4854A",
            markersize=4, lw=1, alpha=0.6, label="vanilla GD (no noise)")
    ax.plot(sgd_path, L(sgd_path), "o-", color="#8B5CF6",
            markersize=4, lw=1, alpha=0.7, label=f"SGD (noise std={noise_std})")
    ax.plot(gd_path[-1],  L(gd_path[-1]),  "D", color="#D4854A",
            markersize=12, markeredgecolor="white", mew=1.2)
    ax.plot(sgd_path[-1], L(sgd_path[-1]), "o", color="#8B5CF6",
            markersize=14, markeredgecolor="white", mew=1.2)
    ax.set_xlabel("θ"); ax.set_ylabel("L(θ)")
    ax.set_title(
        f"{surface_name}   |   "
        f"GD final: L={L(gd_path[-1]):.3f}   "
        f"SGD final: L={L(sgd_path[-1]):.3f}"
    )
    ax.legend(loc="upper right")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


interact(
    plot_sgd,
    surface_name=Dropdown(options=list(SURFACES.keys()),
                          value="local + global min",
                          description="Surface"),
    eta       = FloatSlider(value=0.20, min=0.01, max=2.0, step=0.01,
                            description="η"),
    x_start   = FloatSlider(value=-0.5, min=-2.0, max=8.0, step=0.1,
                            description="θ_start"),
    noise_std = FloatSlider(value=1.0,  min=0.0,  max=3.0, step=0.05,
                            description="noise σ"),
    n_steps   = IntSlider(value=200, min=50, max=1000, step=10,
                            description="steps"),
    seed      = IntSlider(value=0, min=0, max=20, step=1,
                            description="seed"),
);


## 7. Exercises

Try these without looking at the answers.

**E1.** On the convex bowl $L(\theta) = \tfrac{1}{2}(\theta - 3)^2$, the second derivative (curvature) is $L''=1$. Theory says vanilla GD diverges when $\eta > 2/L''=2$. Verify this by binary-searching $\eta$ in the slider until you find the boundary.

**E2.** On `local + global min`, find a $(\eta_\text{mom}, \beta)$ pair where momentum *also* gets stuck at the local minimum. What does that tell you about momentum's "escape capability"?

**E3.** On `wiggly cosine`, both vanilla GD and momentum get trapped immediately. In the SGD section, find a noise level $\sigma$ where SGD has a good chance of finding the global minimum at $\theta \approx 3.14$. SGD is stochastic, so try several seeds — it should *sometimes* succeed, not always. Then push $\sigma$ way too high — what happens to convergence?

**E4.** Modify the code: add a third optimizer, **Adam** (or just **RMSprop**). The update rule for RMSprop is:
$$s_{t+1} = \rho\, s_t + (1-\rho)\,(\nabla L)^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_{t+1}+\epsilon}}\,\nabla L$$
Compare its behavior on the ill-conditioned surfaces.

**E5.** Replace one of the analytical surfaces with a function whose derivative you don't write by hand — use `numpy`'s `np.gradient` to compute a numerical gradient instead. Does the optimizer still work?


## 8. Export an animation

If you want to save a GIF or MP4 of the optimizer trajectory (for a slide, blog, or thesis), here's a self-contained function. It uses `matplotlib.animation` so no extra dependencies beyond what we've already imported.

The cell saves to the current working directory. In Colab the file appears in the left-hand file browser; you can right-click → download.


In [ ]:
import matplotlib.animation as animation


def export_animation(
    surface_name="local + global min",
    eta_gd=0.20,
    eta_mom=0.10,
    beta=0.85,
    x_start=-0.5,
    n_steps=110,
    out_path="gd_vs_momentum.gif",
    fps=15,
):
    """
    Save an animation of vanilla GD vs momentum on a 1D loss surface.

    out_path: ".gif" uses pillow (no extra deps), ".mp4" requires ffmpeg.
    """
    L, dL, (xmin, xmax), _ = SURFACES[surface_name]

    gd_path  = vanilla_gd(dL, x_start, eta_gd, n_steps)
    mom_path = momentum_gd(dL, x_start, eta_mom, beta, n_steps)
    gd_path  = np.clip(gd_path,  xmin - 1, xmax + 1)
    mom_path = np.clip(mom_path, xmin - 1, xmax + 1)

    xs = np.linspace(xmin, xmax, 800)
    ys = L(xs)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(xs, ys, color="gray", lw=2, label="loss L(θ)")
    gd_line,  = ax.plot([], [], "D-", color="#D4854A",
                        markersize=4, lw=1, alpha=0.7, label="vanilla GD")
    mom_line, = ax.plot([], [], "o-", color="#00C9B7",
                        markersize=4, lw=1, alpha=0.7, label="GD + momentum")
    gd_marker,  = ax.plot([], [], "D", color="#D4854A", markersize=12,
                          markeredgecolor="white", mew=1.2)
    mom_marker, = ax.plot([], [], "o", color="#00C9B7", markersize=14,
                          markeredgecolor="white", mew=1.2)

    ax.set_xlabel("θ"); ax.set_ylabel("L(θ)")
    ax.set_title(f"{surface_name}  ·  η_gd={eta_gd}  η_mom={eta_mom}  β={beta}")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.2)

    def init():
        for line in (gd_line, mom_line, gd_marker, mom_marker):
            line.set_data([], [])
        return gd_line, mom_line, gd_marker, mom_marker

    def update(k):
        gd_so_far  = gd_path[: k + 1]
        mom_so_far = mom_path[: k + 1]
        gd_line.set_data(gd_so_far, L(gd_so_far))
        mom_line.set_data(mom_so_far, L(mom_so_far))
        gd_marker.set_data([gd_so_far[-1]], [L(gd_so_far[-1])])
        mom_marker.set_data([mom_so_far[-1]], [L(mom_so_far[-1])])
        return gd_line, mom_line, gd_marker, mom_marker

    n_frames = len(gd_path)
    anim = animation.FuncAnimation(
        fig, update, init_func=init,
        frames=n_frames, interval=1000 / fps, blit=True, repeat=False,
    )

    if out_path.lower().endswith(".gif"):
        anim.save(out_path, writer="pillow", fps=fps)
    elif out_path.lower().endswith(".mp4"):
        anim.save(out_path, writer="ffmpeg", fps=fps)
    else:
        raise ValueError("out_path must end with .gif or .mp4")

    plt.close(fig)
    print(f"saved {out_path}  ({n_frames} frames, {fps} fps)")


# Example: save with the default lecture settings
export_animation(
    surface_name="local + global min",
    eta_gd=0.20, eta_mom=0.10, beta=0.85,
    x_start=-0.5, n_steps=110,
    out_path="gd_vs_momentum.gif",
    fps=15,
)


## 9. Further reading

A short, hand-checked list. All links and DOIs verified; no auto-generated citations.

### Visual / intuitive

- **3Blue1Brown · Deep learning series.** The visual gold standard. Watch chapters 1-3 in order.
  - Chapter 1: *But what is a neural network?* — youtube.com/watch?v=aircAruvnKk
  - Chapter 2: *Gradient descent, how neural networks learn* — youtube.com/watch?v=IHZwWFHWa-w
  - Chapter 3: *What is backpropagation really doing?* — youtube.com/watch?v=Ilg3gGewQ5U
  - Written/interactive companion: 3blue1brown.com/topics/neural-networks

- **Goh, G. (2017). *Why Momentum Really Works.*** Distill, 2(4). https://distill.pub/2017/momentum/  
  Beautifully interactive. Goes well beyond the "ball rolling downhill" intuition; shows where the heavy-ball metaphor breaks. The convex-quadratic analysis here is what the ill-conditioned ravine in our backup slides is really illustrating.

### Textbooks

- **Nielsen, M. (2015). *Neural Networks and Deep Learning.*** Determination Press. Free online at http://neuralnetworksanddeeplearning.com  
  Chapters 1-2 cover gradient descent, backprop, and SGD at a level designed for self-study. A very gentle on-ramp before the Goodfellow book.

- **Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning.*** MIT Press. Free HTML version at https://www.deeplearningbook.org  
  Chapter 4 (Numerical Computation) and Chapter 8 (Optimization for Training Deep Models) are the canonical reference. Chapter 8 in particular covers momentum, Nesterov, AdaGrad, RMSProp, Adam, batch normalization, and second-order methods. Read once, return to often.

### Papers worth reading

- **Sutskever, I., Martens, J., Dahl, G., & Hinton, G. (2013).** *On the importance of initialization and momentum in deep learning.* ICML 2013, pp. 1139-1147. https://proceedings.mlr.press/v28/sutskever13.html  
  Why momentum (and good initialization) matter for training deep networks. Worth reading for the historical context: this is part of the "deep learning works" inflection point.

- **Kingma, D. P., & Ba, J. (2014).** *Adam: A Method for Stochastic Optimization.* arXiv:1412.6980. ICLR 2015. https://arxiv.org/abs/1412.6980  
  The Adam paper. Combines momentum (first moment) with per-parameter learning rates from RMSProp (second moment). The default optimizer in most modern deep learning code; worth understanding from the source.

- **Ruder, S. (2016).** *An overview of gradient descent optimization algorithms.* arXiv:1609.04747. Companion blog: https://www.ruder.io/optimizing-gradient-descent/  
  A practical, opinionated tour of GD variants — Momentum, Nesterov, AdaGrad, AdaDelta, RMSProp, Adam, AdaMax, Nadam. If you want to choose an optimizer for a real project, start here.

### Where to go next

- For the *theory* of optimization: Boyd & Vandenberghe, *Convex Optimization* (free online at https://web.stanford.edu/~boyd/cvxbook/). Hard, but the foundation.
- For *automatic differentiation* (how `dL` is computed in real ML): the JAX or PyTorch documentation and source, or Baydin et al. (2017) "Automatic differentiation in machine learning: a survey" (arXiv:1502.05767).
- For the *modern* training landscape (large batch sizes, learning rate schedules, warmup, weight decay): Loshchilov & Hutter (2019) "Decoupled Weight Decay Regularization" (AdamW, arXiv:1711.05101) is a useful next step.


## 10. Honest caveats

A few things this notebook is *not* showing, in case anyone asks:

- **It's 1D.** Real neural networks have $10^7$ to $10^{12}$ parameters. Most of momentum's practical value comes from navigating ill-conditioned high-dimensional landscapes, not from escaping local minima. In high dimensions, *saddle points* are the dominant non-minimum critical points, not local minima.
- **The hyperparameters are hand-picked.** $\eta=0.20$ and $\beta=0.85$ on the local+global surface aren't sacred — they were chosen to make the contrast visible. Different choices give different stories. That's exactly what the sliders let you explore.
- **Real SGD noise isn't isotropic Gaussian.** It's correlated with the data covariance, and in some regimes it's heavy-tailed. Gaussian is the standard textbook simplification (see Bottou 2010, or the SGD chapter in Goodfellow et al.).
- **Backprop is hidden.** We use analytical derivatives `dL(x)`. In a real network, the gradient comes from automatic differentiation across the computational graph. The optimization logic is unchanged — that's the point of separating the gradient from the optimizer.

If any of these caveats spark a question, that's the right kind of question to bring to a research seminar.
